# PACE-VCF Data Download Notebook

**Purpose:** Download PACE-OCI L3m Surface Reflectance and VIIRS Thermal data for VCF processing.

**Data Period:** DOY 065 2025 through DOY 064 2026 (March 6, 2025 - March 5, 2026)

**Products:**
- PACE-OCI L3m SFREFL (Surface Reflectance)
- PACE-OCI L3m LANDVI (Vegetation Indices) - optional
- VIIRS VNP21A1D (Land Surface Temperature)

**Author:** Melanie Frost
**Date:** 5/28/2026




In [ ]:
## 1. Setup and Configuration

# Install required packages (run once)
# !pip install earthaccess xarray netCDF4 requests tqdm

In [ ]:
import earthaccess
from pathlib import Path
from datetime import datetime, timedelta
from tqdm import tqdm
import os
import json

In [ ]:
# =============================================================================
# CONFIGURATION - MODIFY THESE AS NEEDED
# =============================================================================

# Output directory for downloaded data
OUTPUT_DIR = Path("/explore/nobackup/projects/ilab/data/MODIS/PACE_VCF")

# Year to process
YEAR = 2025

# Day of year range (MODIS-VCF standard: DOY 065 to DOY 064 next year)
START_DOY = 65
END_DOY = 64  # This will be in YEAR + 1

# Calculate actual dates from year and DOY
def doy_to_date(year: int, doy: int) -> str:
    """Convert year and day-of-year to date string (YYYY-MM-DD)."""
    date = datetime(year, 1, 1) + timedelta(days=doy - 1)
    return date.strftime("%Y-%m-%d")

# Generate start and end dates
START_DATE = doy_to_date(YEAR, START_DOY)
END_DATE = doy_to_date(YEAR + 1, END_DOY)

# Display the configuration
print(f"Year: {YEAR}")
print(f"Start: DOY {START_DOY:03d} {YEAR} → {START_DATE}")
print(f"End:   DOY {END_DOY:03d} {YEAR + 1} → {END_DATE}")

# Resolution for PACE L3m products
# Options: "2km", "4km", "0p1deg"
# Recommendation: "4km" or "2km" for best match with MODIS VCF
PACE_RESOLUTION = "2km"  # or "2km" for finest available

# Temporal resolution
# Options: "DAY" (daily), "8D" (8-day), "32D", "MO" (monthly)
TEMPORAL = "DAY" # "8D"

# Bounding box for spatial subset (optional)
# Format: (west, south, east, north) or None for global
BOUNDING_BOX = None  # Global

# Example regional subsets:
# BOUNDING_BOX = (-125, 25, -65, 50)   # CONUS
# BOUNDING_BOX = (-10, 35, 40, 70)     # Europe
# BOUNDING_BOX = (-80, -60, -30, 15)   # South America

# Download options
DOWNLOAD_SFREFL = True    # PACE Surface Reflectance
DOWNLOAD_LANDVI = True    # PACE Vegetation Indices (optional but helpful)
DOWNLOAD_VIIRS = True     # VIIRS Thermal

# Create output directories
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "PACE_SFREFL").mkdir(exist_ok=True)
(OUTPUT_DIR / "PACE_LANDVI").mkdir(exist_ok=True)
(OUTPUT_DIR / "VIIRS_VNP21A1D").mkdir(exist_ok=True)
(OUTPUT_DIR / "logs").mkdir(exist_ok=True)

print(f"Output directory: {OUTPUT_DIR.absolute()}")
print(f"Date range: {START_DATE} to {END_DATE}")
print(f"Resolution: {PACE_RESOLUTION}")
print(f"Temporal: {TEMPORAL}")

In [ ]:
## 2. Earthdata Authentication

# Login to Earthdata
# This will prompt for credentials if not already stored
print("Authenticating with NASA Earthdata...")
auth = earthaccess.login(strategy="interactive")

if auth:
    print("✓ Authentication successful!")
else:
    print("✗ Authentication failed. Please check your credentials.")
    print("  Create an account at: https://urs.earthdata.nasa.gov")

In [ ]:
# 3. Search for Available Data

def search_pace_data(short_name: str, start_date: str, end_date: str, 
                     bbox: tuple = None) -> list:
    """Search for PACE data granules."""
    
    search_params = {
        "short_name": short_name,
        "temporal": (start_date, end_date),
    }
    
    # Only add bounding_box if it's actually defined (not None)
    if bbox is not None:
        search_params["bounding_box"] = bbox
    
    try:
        results = earthaccess.search_data(**search_params)
        return results
    except Exception as e:
        print(f"  Search error: {e}")
        return []

def filter_pace_granules(results: list, resolution: str = "0p1deg", 
                          temporal: str = "DAY") -> list:
    """
    Filter PACE L3m search results by resolution and temporal period.
    
    Parameters
    ----------
    results : list
        earthaccess search results
    resolution : str
        Resolution string: "0p1deg", "2km", "4km"
    temporal : str
        Temporal period: "DAY", "8D", "MO", "R32"
    
    Returns
    -------
    list
        Filtered results
    """
    
    filtered = []
    
    for granule in results:
        try:
            if hasattr(granule, 'data_links'):
                name = str(granule.data_links())
            else:
                name = str(granule)
        except:
            name = str(granule)
        
        temporal_pattern = f"L3m.{temporal}."
        
        if temporal_pattern in name and resolution in name:
            filtered.append(granule)
    
    return filtered

# Search for PACE SFREFL data
print("Searching for PACE-OCI Surface Reflectance data...")
print("  (This may take a moment...)")

# Try different possible short names for PACE L3m SFREFL
pace_sfrefl_names = [
    "PACE_OCI_L3M_SFREFL",
    "PACE_OCI_L3m_SFREFL", 
    "OCI_L3M_SFREFL",
    "PACE_OCI.L3m.SFREFL"
]

sfrefl_results = []
for name in pace_sfrefl_names:
    results = search_pace_data(name, START_DATE, END_DATE, BOUNDING_BOX)
    if results:
        print(f"  Found {len(results)} granules with short_name='{name}'")
        sfrefl_results = results
        SFREFL_SHORT_NAME = name
        break

if not sfrefl_results:
    print("  ⚠ No SFREFL results found. Will try direct URL construction later.")

# %%
# Search for PACE LANDVI data
print("\nSearching for PACE-OCI LANDVI data...")

pace_landvi_names = [
    "PACE_OCI_L3M_LANDVI",
    "PACE_OCI_L3m_LANDVI",
    "OCI_L3M_LANDVI"
]

landvi_results = []
for name in pace_landvi_names:
    results = search_pace_data(name, START_DATE, END_DATE, BOUNDING_BOX)
    if results:
        print(f"  Found {len(results)} granules with short_name='{name}'")
        landvi_results = results
        LANDVI_SHORT_NAME = name
        break

if not landvi_results:
    print("  ⚠ No LANDVI results found.")

# %%
# Search for VIIRS LST data
print("\nSearching for VIIRS Land Surface Temperature data...")

# Build search parameters conditionally
viirs_search_params = {
    "short_name": "VNP21A1D",
    "temporal": (START_DATE, END_DATE),
}

# Only add bounding_box if it's defined
if BOUNDING_BOX:
    viirs_search_params["bounding_box"] = BOUNDING_BOX

viirs_results = earthaccess.search_data(**viirs_search_params)

print(f"  Found {len(viirs_results)} VIIRS LST granules")



In [ ]:
## 4. Download Data

def download_with_progress(results: list, output_dir: Path, desc: str = "Downloading"):
    """Download data with progress bar."""
    
    if not results:
        print(f"  No files to download for {desc}")
        return []
    
    print(f"\n{desc}: {len(results)} files")
    print(f"  Output: {output_dir}")
    
    downloaded = []
    
    try:
        # earthaccess.download handles batching internally
        files = earthaccess.download(results, local_path=str(output_dir))
        downloaded = files
        print(f"  ✓ Downloaded {len(files)} files")
    except Exception as e:
        print(f"  ✗ Download error: {e}")
    
    return downloaded

# %%
# Download PACE SFREFL
if DOWNLOAD_SFREFL and sfrefl_results:
    print(f"\nFiltering SFREFL results for {TEMPORAL} at {PACE_RESOLUTION}...")
    print(f"  Before filtering: {len(sfrefl_results)} granules")
    
    filtered_sfrefl = filter_pace_granules(sfrefl_results, PACE_RESOLUTION, TEMPORAL)
    
    print(f"  After filtering: {len(filtered_sfrefl)} granules")
    
    if filtered_sfrefl:
        sfrefl_files = download_with_progress(
            filtered_sfrefl, 
            OUTPUT_DIR / "PACE_SFREFL",
            "PACE SFREFL"
        )
    else:
        print("  ⚠ No matching granules after filtering!")
        sfrefl_files = []
else:
    print("Skipping PACE SFREFL download")
    sfrefl_files = []

# %%
if DOWNLOAD_LANDVI and landvi_results:
    print(f"\nFiltering LANDVI results for {TEMPORAL} at {PACE_RESOLUTION}...")
    print(f"  Before filtering: {len(landvi_results)} granules")
    
    filtered_landvi = filter_pace_granules(landvi_results, PACE_RESOLUTION, TEMPORAL)
    
    print(f"  After filtering: {len(filtered_landvi)} granules")
    
    if filtered_landvi:
        landvi_files = download_with_progress(
            filtered_landvi,
            OUTPUT_DIR / "PACE_LANDVI", 
            "PACE LANDVI"
        )
    else:
        print("  ⚠ No matching granules after filtering!")
        landvi_files = []
else:
    print("Skipping PACE LANDVI download")
    landvi_files = []
    ls -l
# %%
# Download VIIRS LST
if DOWNLOAD_VIIRS and viirs_results:
    viirs_files = download_with_progress(
        viirs_results,
        OUTPUT_DIR / "VIIRS_LST",
        "VIIRS LST"
    )
else:
    print("Skipping VIIRS download")
    viirs_files = []

In [ ]:
# # %% [markdown]
# # ## 5. Alternative: Direct URL Download
# # 
# # If earthaccess search doesn't find the products, use direct URL construction.

# # %%
# def generate_pace_urls(start_date: str, end_date: str, product: str,
#                        resolution: str, temporal: str = "DAY") -> list:
#     """
#     Generate direct download URLs for PACE L3m data.
    
#     Base URL: https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/
    
#     File naming: PACE_OCI.YYYYMMDD.L3m.{TEMPORAL}.{PRODUCT}.V3.0.{RES}.nc
#     """
    
#     base_url = "https://oceandata.sci.gsfc.nasa.gov/ob/getfile"
    
#     # Map resolution to filename format
#     res_map = {
#         "4km": "4km",
#         "9km": "9km", 
#         "0.1deg": "0.1deg",
#         "1deg": "1deg"
#     }
#     res_str = res_map.get(resolution, resolution)
    
#     urls = []
#     current = datetime.strptime(start_date, "%Y-%m-%d")
#     end = datetime.strptime(end_date, "%Y-%m-%d")
    
#     while current <= end:
#         date_str = current.strftime("%Y%m%d")
        
#         # Construct filename
#         fname = f"PACE_OCI.{date_str}.L3m.{temporal}.{product}.V3.0.{res_str}.nc"
#         url = f"{base_url}/{fname}"
#         urls.append((current.strftime("%Y-%m-%d"), fname, url))
        
#         # Advance date based on temporal resolution
#         if temporal == "DAY":
#             current += timedelta(days=1)
#         elif temporal == "8D":
#             current += timedelta(days=8)
#         else:  # Monthly
#             if current.month == 12:
#                 current = current.replace(year=current.year + 1, month=1, day=1)
#             else:
#                 current = current.replace(month=current.month + 1, day=1)
    
#     return urls

# # %%
# # Generate URLs for manual download if needed
# print("Generating direct download URLs (for reference/backup)...")

# sfrefl_urls = generate_pace_urls(START_DATE, END_DATE, "SFREFL", PACE_RESOLUTION, TEMPORAL)
# print(f"  SFREFL: {len(sfrefl_urls)} URLs generated")

# landvi_urls = generate_pace_urls(START_DATE, END_DATE, "LANDVI", PACE_RESOLUTION, TEMPORAL)
# print(f"  LANDVI: {len(landvi_urls)} URLs generated")

# # Show first few URLs as example
# print("\nExample SFREFL URLs:")
# for date, fname, url in sfrefl_urls[:3]:
#     print(f"  {date}: {fname}")

# # %%
# # Save URLs to file for wget/curl download
# def save_url_list(urls: list, output_file: Path):
#     """Save URL list to file."""
#     with open(output_file, 'w') as f:
#         for date, fname, url in urls:
#             f.write(f"{url}\n")
#     print(f"  Saved to {output_file}")

# save_url_list(sfrefl_urls, OUTPUT_DIR / "logs" / "sfrefl_urls.txt")
# save_url_list(landvi_urls, OUTPUT_DIR / "logs" / "landvi_urls.txt")

# # %%
# # Create wget download script
# def create_wget_script(urls: list, output_file: Path, output_dir: Path):
#     """Create bash script for wget download."""
    
#     with open(output_file, 'w') as f:
#         f.write("#!/bin/bash\n")
#         f.write("# PACE Data Download Script\n")
#         f.write("# Generated by PACE-VCF Download Notebook\n\n")
#         f.write("# Setup: Create .netrc file with Earthdata credentials:\n")
#         f.write("# machine urs.earthdata.nasa.gov login YOUR_USERNAME password YOUR_PASSWORD\n\n")
#         f.write(f"OUTPUT_DIR=\"{output_dir}\"\n")
#         f.write("mkdir -p $OUTPUT_DIR\n\n")
#         f.write("cd $OUTPUT_DIR\n\n")
        
#         for date, fname, url in urls:
#             f.write(f"# {date}\n")
#             f.write(f"wget -nc -c --load-cookies ~/.urs_cookies ")
#             f.write(f"--save-cookies ~/.urs_cookies ")
#             f.write(f"--auth-no-challenge=on ")
#             f.write(f"--content-disposition \"{url}\"\n\n")
    
#     print(f"  Created wget script: {output_file}")

# create_wget_script(sfrefl_urls, OUTPUT_DIR / "logs" / "download_sfrefl.sh", 
#                    OUTPUT_DIR / "PACE_SFREFL")
# create_wget_script(landvi_urls, OUTPUT_DIR / "logs" / "download_landvi.sh",
#                    OUTPUT_DIR / "PACE_LANDVI")

In [ ]:
## 6. Verify Downloads

def verify_downloads(data_dir: Path, expected_pattern: str = "*.nc") -> dict:
    """Verify downloaded files and report statistics."""
    
    files = list(data_dir.glob(expected_pattern))
    
    # Also check for HDF5 files (VIIRS)
    h5_files = list(data_dir.glob("*.h5"))
    files.extend(h5_files)
    
    total_size = sum(f.stat().st_size for f in files) / (1024**3)  # GB
    
    result = {
        "directory": str(data_dir),
        "file_count": len(files),
        "total_size_gb": round(total_size, 2),
        "files": [f.name for f in sorted(files)[:10]],  # First 10
    }
    
    return result

# %%
print("\n" + "="*60)
print("DOWNLOAD VERIFICATION")
print("="*60)

# Check each data directory
for subdir in ["PACE_SFREFL", "PACE_LANDVI", "VIIRS_LST"]:
    dir_path = OUTPUT_DIR / subdir
    if dir_path.exists():
        stats = verify_downloads(dir_path)
        print(f"\n{subdir}:")
        print(f"  Files: {stats['file_count']}")
        print(f"  Size: {stats['total_size_gb']} GB")
        if stats['files']:
            print(f"  Sample files: {stats['files'][:3]}")
    else:
        print(f"\n{subdir}: Directory not found")



In [ ]:
# %% [markdown]
# ## 7. Create Download Summary

# %%
# Save download summary
summary = {
    "download_date": datetime.now().isoformat(),
    "configuration": {
        "year": YEAR,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "resolution": PACE_RESOLUTION,
        "temporal": TEMPORAL,
        "bounding_box": BOUNDING_BOX
    },
    "products": {
        "PACE_SFREFL": verify_downloads(OUTPUT_DIR / "PACE_SFREFL"),
        "PACE_LANDVI": verify_downloads(OUTPUT_DIR / "PACE_LANDVI"),
        "VIIRS_LST": verify_downloads(OUTPUT_DIR / "VIIRS_LST")
    }
}

# Save to JSON
summary_file = OUTPUT_DIR / "logs" / "download_summary.json"
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\nDownload summary saved to: {summary_file}")

# %%
print("\n" + "="*60)
print("DOWNLOAD COMPLETE")
print("="*60)
print(f"\nData saved to: {OUTPUT_DIR.absolute()}")
print("\nNext steps:")
print("  1. Verify all expected files were downloaded")
print("  2. Run Notebook 2 to create composites")
print("  3. If downloads failed, use the wget scripts in logs/")

# %% [markdown]
# ## 8. Troubleshooting
# 
# ### If earthaccess search returns no results:
# 1. Check that the product short_name is correct
# 2. Try the direct URL download method (Section 5)
# 3. Use the wget scripts generated in the `logs/` folder
# 
# ### If authentication fails:
# 1. Verify your Earthdata credentials at https://urs.earthdata.nasa.gov
# 2. Check that you have approved the necessary applications
# 3. Try logging in via browser first
# 
# ### If downloads are slow:
# 1. Use the wget scripts for parallel downloads
# 2. Consider downloading a regional subset first
# 3. Try during off-peak hours (nights/weekends US time)